# Glidepath — Monte Carlo retirement projection

A quick tour of the `glidepath` core engine: simulate a portfolio through an
equity **glidepath**, with randomized returns, sequence risk, and stochastic
inflation, then look at the results with the shared Plotly charts.

Run inside the workspace venv:

```bash
uv run --with ipykernel jupyter lab   # or: uv run jupyter nbconvert --execute notebooks/demo.ipynb
```

> Educational model — not financial advice. Plug in your own ETF / IKZE numbers.

In [1]:
from glidepath import SimulationParams, simulate, summarize
from glidepath import viz

# A baseline saver: 30 today, retires at 65, plans to 95.
params = SimulationParams(
    start_age=30,
    retirement_age=65,
    end_age=95,
    initial_balance=50_000,
    annual_contribution=18_000,
    annual_spending=40_000,
    other_retirement_income=12_000,
    n_paths=10_000,
    seed=42,
)

result = simulate(params)
summarize(result)

Summary(success_probability=0.896, median_terminal_real=1072533.851227322, p10_terminal_real=0.0, p90_terminal_real=3864828.574011107, median_depletion_age=90.0)

## Fan chart — the spread of outcomes

The median line is the "typical" path; the shaded bands are the P10–P90 and
P25–P75 ranges across all simulated paths, in **today's money**.

In [2]:
viz.fan_chart(result, real=True, currency="zł ")

## The glidepath itself, and the terminal-wealth distribution

In [3]:
viz.glidepath_chart(result).show()
viz.terminal_histogram(result, real=True, currency="zł ").show()

## Compare glidepath styles

Because the core takes plain dataclasses, sweeping scenarios is trivial. Here we
compare how the equity glide affects the probability the money lasts.

In [4]:
from glidepath import Glidepath

scenarios = {
    "Aggressive hold (90% flat)": Glidepath(style="constant", start_equity=0.90),
    "Classic glide (90% -> 40%)": Glidepath(style="linear", start_equity=0.90, end_equity=0.40),
    "Conservative (60% -> 20%)": Glidepath(style="linear", start_equity=0.60, end_equity=0.20),
    "Age rule (110 - age)": Glidepath(style="age_rule", rule_base=110),
}

for name, gp in scenarios.items():
    res = simulate(params.with_(glidepath=gp))
    s = summarize(res)
    print(f"{name:30s}  lasts {s.success_probability*100:5.1f}%  "
          f"median end (real) zł {s.median_terminal_real:,.0f}")

Aggressive hold (90% flat)      lasts  87.4%  median end (real) zł 1,647,943
Classic glide (90% -> 40%)      lasts  89.6%  median end (real) zł 1,072,534
Conservative (60% -> 20%)       lasts  88.3%  median end (real) zł 558,757
Age rule (110 - age)            lasts  89.2%  median end (real) zł 690,264
